# Intent classifier: TF-IDF + Logistic Regression

**Goal:** given a chat message, predict what the user wants:
`find_posts`, `summarize_post`, `create_post`, `site_help`, `greeting`, `out_of_scope`.

**Data**
- `data/intents.csv`: 180 training messages (30 per intent)
- `data/intents_test.csv`: 72 messages written separately, **used only once, at the end**

**Plan:** load → split → dumb baseline → model → tune → evaluate → confidence threshold → final test → save.

Run the cells top to bottom (kernel: `ai-service/.venv`).

## 1. Load the data
`RANDOM_STATE` fixes the randomness, so every run gives the same split and the same results.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, MODELS = ROOT / "data", ROOT / "models"

train_df = pd.read_csv(DATA / "intents.csv")
test_df = pd.read_csv(DATA / "intents_test.csv")   # locked away until step 8

print("train:", train_df.shape, "| test:", test_df.shape)
train_df.label.value_counts()

train: (419, 2) | test: (84, 2)


label
find_posts        60
post_report       60
summarize_post    60
greeting          60
out_of_scope      60
create_post       60
site_help         59
Name: count, dtype: int64

## 2. Split training data into train + validation
We tune on **validation**, never on the test set. `stratify` keeps 6 equal classes in both parts.
With 180 rows, 20% validation = 36 messages (6 per intent).

In [2]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df.text, train_df.label,
    test_size=0.2, stratify=train_df.label, random_state=RANDOM_STATE,
)
print("train:", len(X_train), "| validation:", len(X_val))
y_val.value_counts()

train: 335 | validation: 84


label
greeting          12
site_help         12
summarize_post    12
find_posts        12
create_post       12
out_of_scope      12
post_report       12
Name: count, dtype: int64

## 3. Dumb baseline
Always predicts the most common class. With 6 balanced classes that's about 1/6 ≈ 17%.
**Any real model must beat this clearly**, otherwise something is wrong.

In [3]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy accuracy: {accuracy_score(y_val, dummy.predict(X_val)):.0%}")

Dummy accuracy: 14%


## 4. First real model
A `Pipeline` chains two steps so they always run together:
1. **TfidfVectorizer** turns text into numbers: each word gets a weight (frequent in this message, rare overall = important).
2. **LogisticRegression** learns which weights point to which intent.

`ngram_range=(1, 2)` also uses word pairs like *"how do"* or *"for me"*, which matter a lot for these intents.

In [4]:
first = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])
first.fit(X_train, y_train)
print(f"Validation accuracy: {accuracy_score(y_val, first.predict(X_val)):.0%}")

Validation accuracy: 81%


## 5. Tune the settings with cross-validation
`GridSearchCV` tries every combination below. For each, `StratifiedKFold(5)` trains 5 times on 4/5 of the
training data and scores on the other 1/5. That's more reliable than one small validation split.

- **word** n-grams: whole words
- **char_wb** n-grams: pieces of words (2–5 letters). Helps with typos (*shw, abt*) and Hinglish spellings.
- **C**: how strongly the model fits the data. Small C = simpler model, large C = fits training data more closely.

We score with **macro F1**: the average F1 over the 6 intents, so every intent counts equally.

In [5]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000)),
])
grid = {
    "tfidf__analyzer": ["word", "char_wb"],
    "tfidf__ngram_range": [(1, 1), (1, 2), (2, 4), (2, 5)],
    "clf__C": [1, 10, 100],
}
search = GridSearchCV(
    pipe, grid, scoring="f1_macro",
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
)
search.fit(X_train, y_train)

print("Best settings:", search.best_params_)
print(f"Best cross-validation macro F1: {search.best_score_:.0%}")

results = pd.DataFrame(search.cv_results_)
cols = ["param_tfidf__analyzer", "param_tfidf__ngram_range", "param_clf__C", "mean_test_score"]
results[cols].sort_values("mean_test_score", ascending=False).head(8)

Best settings: {'clf__C': 1, 'tfidf__analyzer': 'word', 'tfidf__ngram_range': (1, 2)}
Best cross-validation macro F1: 76%


,param_tfidf__analyzer,param_tfidf__ngram_range,param_clf__C,mean_test_score
1,word,"(1, 2)",1,0.756185
14,char_wb,"(2, 4)",10,0.755095
17,word,"(1, 2)",100,0.753341
22,char_wb,"(2, 4)",100,0.752354
23,char_wb,"(2, 5)",100,0.752075
15,char_wb,"(2, 5)",10,0.751828
9,word,"(1, 2)",10,0.747267
7,char_wb,"(2, 5)",1,0.746661


## 6. Evaluate on validation
- **precision**: of the messages predicted as X, how many really were X
- **recall**: of the real X messages, how many we found
- **confusion matrix**: rows = true intent, columns = predicted. Everything off the diagonal is a mistake.

**Reading the wrong predictions teaches you the most.** Ask: is the label unclear? Are examples missing?

In [6]:
best = search.best_estimator_
val_pred = best.predict(X_val)
print(classification_report(y_val, val_pred, zero_division=0))
pd.crosstab(y_val, val_pred, rownames=["true"], colnames=["predicted"])

                precision    recall  f1-score   support

   create_post       0.71      1.00      0.83        12
    find_posts       0.89      0.67      0.76        12
      greeting       0.75      1.00      0.86        12
  out_of_scope       1.00      0.42      0.59        12
   post_report       0.85      0.92      0.88        12
     site_help       0.77      0.83      0.80        12
summarize_post       0.91      0.83      0.87        12

      accuracy                           0.81        84
     macro avg       0.84      0.81      0.80        84
  weighted avg       0.84      0.81      0.80        84



predicted,create_post,find_posts,greeting,out_of_scope,post_report,site_help,summarize_post
true,,,,,,,
create_post,12,0,0,0,0,0,0
find_posts,2,8,1,0,1,0,0
greeting,0,0,12,0,0,0,0
out_of_scope,1,0,3,5,0,2,1
post_report,0,0,0,0,11,1,0
site_help,1,0,0,0,1,10,0
summarize_post,1,1,0,0,0,0,10


In [7]:
wrong = pd.DataFrame({"text": X_val, "true": y_val, "predicted": val_pred})
wrong = wrong[wrong.true != wrong.predicted]
print(len(wrong), "wrong out of", len(X_val))
wrong

16 wrong out of 84


,text,true,predicted
323,why was i logged out,site_help,post_report
235,koi travel wala post dikhao,find_posts,create_post
135,play some music,out_of_scope,greeting
137,write my college essay,out_of_scope,create_post
156,lol,out_of_scope,greeting
373,how do i lose weight fast,out_of_scope,site_help
64,shorten the healthy eating blog for me,summarize_post,create_post
227,blog posts from last month,find_posts,post_report
5,are there blogs related to fitness,find_posts,greeting
386,hmm,out_of_scope,greeting


## 7. Confidence threshold
`predict_proba` gives a probability per intent. If the model isn't sure (top probability below a threshold),
the bot should treat the message as `out_of_scope` instead of guessing.

Look at how sure the model is when it's **right** vs **wrong**, then compare thresholds:
- **answered**: share of messages the bot would act on
- **accuracy**: how often it's right when it does act, with low-confidence messages counted as `out_of_scope`

In [8]:
val_proba = best.predict_proba(X_val)
val_conf = val_proba.max(axis=1)
right = val_pred == y_val.to_numpy()
print(f"avg confidence when right: {val_conf[right].mean():.2f}")
if (~right).any():
    print(f"avg confidence when wrong: {val_conf[~right].mean():.2f}")

def with_threshold(pred, conf, t):
    return np.where(conf >= t, pred, "out_of_scope")

rows = []
for t in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6]:
    p = with_threshold(val_pred, val_conf, t)
    rows.append({"threshold": t, "answered": (val_conf >= t).mean(),
                 "accuracy": accuracy_score(y_val, p), "macro_f1": f1_score(y_val, p, average="macro")})
pd.DataFrame(rows).style.format({"answered": "{:.0%}", "accuracy": "{:.0%}", "macro_f1": "{:.0%}"})

avg confidence when right: 0.36
avg confidence when wrong: 0.26


,threshold,answered,accuracy,macro_f1
0,0.000000,100%,81%,80%
1,0.200000,86%,79%,79%
2,0.300000,55%,60%,64%
3,0.400000,26%,38%,41%
4,0.500000,15%,27%,27%
5,0.600000,4%,15%,8%


Pick the threshold with the best macro F1 on **validation** (not on the test set). With 6 classes and
little data, the model's probabilities are spread out, so a good threshold is often lower than you'd expect.

In [9]:
THRESHOLD = max(rows, key=lambda r: r["macro_f1"])["threshold"]
print("Chosen threshold:", THRESHOLD)

Chosen threshold: 0.0


## 8. Final test, run once
Retrain the best settings on **all** 180 training rows, then score your 72 test messages.
This is the honest number. Write it down even if it's lower than validation: that tells you how
well the model handles wording it has never seen.

In [10]:
final = search.best_estimator_.fit(train_df.text, train_df.label)

X_test, y_test = test_df.text, test_df.label
test_pred = final.predict(X_test)
test_conf = final.predict_proba(X_test).max(axis=1)
test_pred_t = with_threshold(test_pred, test_conf, THRESHOLD)

print(f"Dummy baseline:                 {accuracy_score(y_test, dummy.predict(X_test)):.0%}")
print(f"Test accuracy:                  {accuracy_score(y_test, test_pred):.0%}  (macro F1 {f1_score(y_test, test_pred, average='macro'):.0%})")
print(f"Test accuracy, threshold {THRESHOLD}:  {accuracy_score(y_test, test_pred_t):.0%}  (macro F1 {f1_score(y_test, test_pred_t, average='macro'):.0%})")
print()
print(classification_report(y_test, test_pred, zero_division=0))
pd.crosstab(y_test, test_pred, rownames=["true"], colnames=["predicted"])

Dummy baseline:                 14%
Test accuracy:                  87%  (macro F1 87%)
Test accuracy, threshold 0.0:  87%  (macro F1 87%)

                precision    recall  f1-score   support

   create_post       1.00      0.83      0.91        12
    find_posts       0.92      0.92      0.92        12
      greeting       0.92      0.92      0.92        12
  out_of_scope       0.71      0.83      0.77        12
   post_report       1.00      0.92      0.96        12
     site_help       0.82      0.75      0.78        12
summarize_post       0.79      0.92      0.85        12

      accuracy                           0.87        84
     macro avg       0.88      0.87      0.87        84
  weighted avg       0.88      0.87      0.87        84



predicted,create_post,find_posts,greeting,out_of_scope,post_report,site_help,summarize_post
true,,,,,,,
create_post,10,0,0,0,0,0,2
find_posts,0,11,0,0,0,1,0
greeting,0,0,11,1,0,0,0
out_of_scope,0,0,1,10,0,0,1
post_report,0,0,0,0,11,1,0
site_help,0,1,0,2,0,9,0
summarize_post,0,0,0,1,0,0,11


In [11]:
test_wrong = pd.DataFrame({"text": X_test, "true": y_test, "predicted": test_pred, "confidence": test_conf.round(2)})
test_wrong = test_wrong[test_wrong.true != test_wrong.predicted].sort_values("true")
print(len(test_wrong), "wrong out of", len(X_test))
pd.set_option("display.max_colwidth", 90)
test_wrong

11 wrong out of 84


,text,true,predicted,confidence
29,Write this in a professional tone for a business audience,create_post,summarize_post,0.30
33,I have the topic already: healthy habits for students. Please write the post,create_post,summarize_post,0.27
9,Can I see what Rahul has published recently?,find_posts,site_help,0.37
58,What's up?,greeting,out_of_scope,0.22
63,Can you calculate 847 divided by 13?,out_of_scope,greeting,0.19
64,Tell me the latest news about the election,out_of_scope,summarize_post,0.31
75,The admin sent my cooking post back. What do they want me to change?,post_report,site_help,0.24
36,The editor keeps showing a 500 error when I try to publish. What should I do?,site_help,out_of_scope,0.25
40,Which image formats are allowed on the blog?,site_help,find_posts,0.23
41,What is the difference between writer and admin roles?,site_help,out_of_scope,0.33


## 9. Save the model
`joblib` stores the whole pipeline (vectorizer + classifier) in one file. The FastAPI service will load it
for `POST /intent`. We save the threshold and scores with it, so we know later which model this was.

In [12]:
MODELS.mkdir(exist_ok=True)
bundle = {
    "pipeline": final,
    "threshold": THRESHOLD,
    "labels": list(final.classes_),
    "params": search.best_params_,
    "test_accuracy": float(accuracy_score(y_test, test_pred)),
    "test_macro_f1": float(f1_score(y_test, test_pred, average="macro")),
}
path = MODELS / "intent_tfidf_lr.joblib"
joblib.dump(bundle, path)
print("saved", path.name, f"({path.stat().st_size / 1024:.0f} KB)")

saved intent_tfidf_lr.joblib (145 KB)


## 10. Try your own messages
Change the list and re-run this cell.

In [13]:
def predict_intent(text):
    proba = final.predict_proba([text])[0]
    i = proba.argmax()
    intent = final.classes_[i] if proba[i] >= THRESHOLD else "out_of_scope"
    return intent, round(float(proba[i]), 2)

for msg in ["koi achha post hai photography pe?",
            "can you write a post about diwali with my photo",
            "how do I add an image to my post",
            "what's the score of the match",
            "tldr this article pls"]:
    print(f"{msg!r:52} -> {predict_intent(msg)}")

'koi achha post hai photography pe?'                 -> ('find_posts', 0.22)
'can you write a post about diwali with my photo'    -> ('create_post', 0.62)
'how do I add an image to my post'                   -> ('site_help', 0.55)
"what's the score of the match"                      -> ('out_of_scope', 0.34)
'tldr this article pls'                              -> ('summarize_post', 0.67)


## Experiment log
| Date | Change | CV macro F1 | Test accuracy | Notes |
|---|---|---|---|---|
| | first run | | | |

**Ideas to try next:** add examples for the intents that get confused, try `class_weight`, try `LinearSVC`,
combine word + char features (`FeatureUnion`), then a small transformer (next notebook).